In [0]:
# CELL: Install Python-Dotenv
#-----------------------------------------------------------

# 1. Install Python-Dotenv
%pip install python-dotenv

# 2. Verify Installation
try:
    from dotenv import load_dotenv
    print("Dotenv is installed and working correctly.")
except ImportError:
    raise ImportError(
        "Dotenv failed to install.  ", 
        "Please rerun this cell",
    )

In [0]:
# CELL: Import Required Libraries
#-----------------------------------------------------------

from dotenv import load_dotenv
import os
from pyspark.sql.functions import col, sum, avg, count, month, year, round, desc

In [0]:
# CELL: Load Environment Variables
#-----------------------------------------------------------

# 1. Load environment variables from file
load_dotenv()

# 2. Assign storage acount name and key vars from env
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")

# 3. Verify env read successfully
print(f"storage_account_name = {storage_account_name}")

In [0]:
# CELL: Configure spark settings
#-----------------------------------------------------------

# 1. AUTO-CREATE-FIX: Configuration auto-creates the 'bronze' container if it is missing
spark.conf.set(
    f"fs.azure.createRemoteFileSystemDuringInitialization", "true"
)

# 2. Connect to storage account
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key,
)

print("Configuration loaded & Auto-create enabled")

In [0]:
# CELL: Assign constants
#-----------------------------------------------------------

# 1. Assign zones 
ZN_TRANSACTIONS = "transactions"
ZN_UNITS = "units"

# 1. Assign Silver & Gold
SILVER = "silver"
GOLD = "gold"

# 2. Assign medallion paths
MEDALLION_PATHS = {medallion: f"abfss://{medallion}@{storage_account_name}.dfs.core.windows.net/" for medallion in [SILVER, GOLD]}

SILVER_PATH = MEDALLION_PATHS[SILVER]
GOLD_PATH = MEDALLION_PATHS[GOLD]

# 3. Assign medallion paths
SILVER_DATA_PATHS = {zn_data: f"{SILVER_PATH}{zn_data}" for zn_data in [ZN_TRANSACTIONS, ZN_UNITS]}


In [0]:
# CELL: Load Silver Input Dataframes
#-----------------------------------------------------------

# 1. Assign Silver Dataframes mapped to appropriate zone
SILVER_DF_MAPPINGS = {
    zone: spark.read.format("delta").load(data_path) 
    for zone, data_path 
    in SILVER_DATA_PATHS.items()
}

In [0]:
# CELL: Announce Configurations Setup and Data Loaded
#-----------------------------------------------------------

print(f"{SILVER} Data Loaded & {GOLD} Container Configured")

In [0]:
# CELL: Assign as individual DataFrames
#-----------------------------------------------------------

df_silver_txns = SILVER_DF_MAPPINGS[ZN_TRANSACTIONS]
df_silver_unit = SILVER_DF_MAPPINGS[ZN_UNITS]

In [0]:
# CELL: Gold Table 1 - Monthly Sales Trends
#-----------------------------------------------------------
# Goal: Track Total Amount and Count of Sales/Month

gold_sales_overview = df_silver_txns.groupBy(
    year("transaction_date").alias("year"),
    month("transaction_date").alias("month"),
).agg(
    sum("price").alias("total_sales_aed"),
    count("transaction_id").alias("total_transactions"),
    avg("price_per_sqft").alias("avg_price_sqft"),
).orderBy("year", "month")

# Round the numbers for cleaner reports
gold_sales_overview = gold_sales_overview.withColumn("total_sales_aed", round(col("total_sales_aed"), 2))
gold_sales_overview = gold_sales_overview.withColumn("avg_price_sqft", round(col("avg_price_sqft"), 2))

print("Sales Overview Created.")

display(gold_sales_overview)

In [0]:
# CELL: Gold Table 2 - Area Performance
#-----------------------------------------------------------
# Goal: Which areas have the highest sales volume?

gold_area_stats = df_silver_txns.groupBy("area_name").agg(
    sum("price").alias("total_volume_aed"),
    count("transaction_id").alias("transaction_count"),
    avg("price_per_sqft").alias("avg_sqft_price"),
).orderBy(desc("total_volume_aed"))

# Cleanup
gold_area_stats = gold_area_stats.withColumn("total_volume_aed", round(col("total_volume_aed"), 0))
gold_area_stats = gold_area_stats.withColumn("avg_sqft_price", round(col("avg_sqft_price"), 2))

print("Area Stats Created.")

display(gold_area_stats.limit(10))

In [0]:
# CELL: Gold Table 3 - Supply & Inventory
#-----------------------------------------------------------
# Goal: Count units by Bedroom type (Supply Side)

gold_supply_stats = df_silver_unit.groupBy(
    "project_number",
    "room_type",
    "bedrooms",
).agg(
    count("unit_number").alias("unit_count"),
    avg("unit_area").alias("avg_unit_area"),
).orderBy(desc("unit_count"))

print("Supply Stats Created.")
display(gold_supply_stats.limit(10))

In [0]:
# CELL: Write to Gold Layer
#-----------------------------------------------------------

print("Writing Gold Tables...")

# 1. Save Sales Overview
gold_sales_overview.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}sales_overview")

# 2. Save Area Stats
gold_area_stats.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}area_stats")

# 3. Save Supply Stats
gold_supply_stats.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}supply_stats")

print("SUCCESS: Gold Layer (Analytics Ready) Created!")

In [0]:
# CELL:Verification
#-----------------------------------------------------------

try:
    check_df = spark.read.format("delta").load(f"{GOLD_PATH}sales_overview")
    count = check_df.count()

    print(f"VERIFIED: Gold Sales Table contains {count} monthly records")

    if count > 0:
        display(check_df.limit(5))
    else:
        print("WARNING: Table is empty.")
        
except Exception as e:
    print(f"Error: {e}")